# VCNN on Kaggle (GPU)

This notebook clones the repo, installs the extra dependencies, runs a fast smoke test, then trains.

**Setup (do this first):**
1. Settings -> Accelerator -> **GPU T4 x2 / P100**.
2. Settings -> **Internet -> On**  (required for `pip install`; one-time phone verification).

**To use your own data:** upload your `.h5` as a Kaggle Dataset and set `DATA` below to its path (e.g. `/kaggle/input/<dataset>/file.h5`).

In [ ]:
# 1) Clone the repo (Kaggle notebooks usually start empty).
#    Replace the URL with YOUR GitHub fork.
import os
if not os.path.exists('/kaggle/working/vcnn'):
    !git clone https://github.com/<your-username>/3d-phenomics.git /kaggle/working/3d-phenomics
%cd /kaggle/working/3d-phenomics/vcnn

In [ ]:
# 2) Sanity: GPU + torch are preinstalled. Install ONLY the missing extras.
#
# NOTE: Kaggle disables internet by default. If you see
#      "ERROR: Could not find a version that satisfies the requirement ..."
#      it means internet is OFF -> enable it in Settings -> Internet -> On.
# This helper first checks whether a package is already importable and only
# then tries to install, so it works even when some deps are preinstalled.
import importlib
import subprocess
import sys


def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
        print(f"[OK]   {import_name} already available")
    except ImportError:
        print(f"[..]   installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        print(f"[DONE] {import_name}")


import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

for pkg, imp in [("pytables", "tables"), ("numexpr", "numexpr"),
                 ("h5py", "h5py"), ("scikit-learn", "sklearn")]:
    ensure(pkg, imp)

In [ ]:
# 3) Fast smoke test: validates model + data pipeline end-to-end.
!python scripts/smoke_test.py

In [ ]:
# 4) Train on the bundled demo data (small; just to confirm it learns).
#    For a real run, point --data at your Kaggle Dataset .h5 and raise --epochs.
!python scripts/train.py --data data/x_y1_y2_idx_33_withoutnoise.h5 --model vcnn_demo --epochs 5

### Resume after a session cut
Kaggle sessions are limited (~12h GPU, weekly quota). Checkpoints are saved to `/kaggle/working/checkpoints`.
Re-run the same cell with `--resume <epoch>` to continue:
```
!python scripts/train.py --data data/x_y1_y2_idx_33_withoutnoise.h5 --model vcnn_demo --epochs 20 --resume 5
```